# Importing libraries

In [1]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv, find_dotenv
from google import genai
from google.genai import types
import google.genai.errors 
import itertools
import json
import time
from pathlib import Path
from pydantic import BaseModel
from typing import Tuple, Optional
import re
import inspect
from google.genai.types import HttpOptions

In [2]:
inspect.ismodule(google.genai.errors.APIError.status)

False

In [3]:
# Loading dot env file
load_dotenv(find_dotenv())

True

In [4]:
# Displaying all columns of df
pd.set_option("display.max_columns", None)

# Importing data

In [5]:
file_path = r"Uncleaned Jobs and skills database 15Sept25.parquet"
df = pd.read_parquet(file_path)
df.head()

,Title,Code,Description,Sector Name,Level,Maximum Notational Hours,Minimum Notational Hours,Version,Originally Approved,Valid Till,Awarding Body,Certifying Bodies,Proposed Occupation,Progression Pathway,Qualifcation Type,Adopted Qualifcation,Training Delivery Hours,NQR code,Qf link,pdf_number,nco_code_full,nco_4digit_code,pc_text
0,Fire Safety Technician (Oil & Gas),2020/HYC/HSSCI/3611,The main responsibility of the fire safety tec...,Hydrocarbon,Level 4,450 Hours,450 Hours,Version,17 Nov 2022,16 Nov 2025,Hydrocarbon Sector Skill Council (HSSCI),Hydrocarbon Sector Skill Council,"Management of Health, Safety and Environment (...",Senior Fire Safety Technician,General Qualification,N.A.,"{""Theory"":""120"",""Practical"":""240"",""Employabili...",2020/HYC/HSSCI/3611,https://www.nqr.gov.in/qualification/file/STT-...,0,[3119.0800],[3119],[maintain fire safety equipment as per mainten...
1,Hindi Typist,2020/OAFM/MEPSC/03792,"The Hindi Typist, is responsible for formattin...",Management,Level 4,450 Hours,390 Hours,Version,17 Nov 2022,17 Nov 2025,Management & Entrepreneurship and Professional...,Management Entrepreneurship and Professional S...,Office Support,Multi-functional Office Executive,General Qualification,N.A.,"{""Theory"":""150"",""Practical"":""180"",""Employabili...",2020/OAFM/MEPSC/03792,https://www.nqr.gov.in/qualification/file/QFil...,1,[4131.9900],[4131],[access specified data or information using sp...
2,Certificate Course in Coding Skills,2020/ITES/ASAP/03802,Individuals at this job are responsible for de...,IT-ITeS,Level 5,270 Hours,270 Hours,Version,25 Jun 2020,01 Mar 2026,"Additional Skill Acquisition Programme, Govern...","Additional Skill Acquisition Programme, Govern...",Software Engineer /Project Engineer,"VERTICAL PROGRESSION \nEngineer Trainee, Proje...","Future Skills Qualification,General Qualification",N.A.,"{""Theory"":""36"",""Practical"":""204"",""Employabilit...",2020/ITES/ASAP/03802,https://www.nqr.gov.in/qualification/file/Q%20...,2,[2512.0800],[2512],None
3,Transit and Self-Loading Mixer Operator,2020/CON/IESC/3881,Transit and Self-Loading Mixer operator drives...,Infrastructure,Level 4,390 Hours,390 Hours,Version,17 Oct 2019,17 Oct 2022,Infrastructure Equipment Sector Skill Council,Infrastructure Equipment Sector Council,Transit and Self-Loading Mixer Operator,Senior Transit and Self-Loading mixer operator,General Qualification,N.A.,"{""Theory"":""90"",""Practical"":""150"",""Employabilit...",2020/CON/IESC/3881,https://nqr.gov.in/sites/default/files/QF%20-I...,3,[8114.0300],[8114],[support in administering basic firstaid and r...
4,AI – Data Architect,2020/ITES/ITSSC/04327,Individuals at this job must be responsible fo...,IT-ITeS,Level 7,750 Hours,660 Hours,Version,19 Dec 2018,22 Sep 2025,IT-ITeS Sector Skills Council NASSCOM (SSC NAS...,IT-ITeS SSC NASSCOM,Artificial Intelligence and Big Data Analytics,"Solutions Architect, Senior Database Administr...",Upskilling Qualification,N.A.,"{""Theory"":""180"",""Practical"":""330"",""Employabili...",2020/ITES/ITSSC/04327,https://www.nqr.gov.in/qualification/file/SSC%...,4,None,None,[encourage team members with diverse view poin...


In [6]:
# Loading API keys
api_keys = [os.getenv("gemini_api_tm_per"), os.getenv("gemini_api_tm_ceew") ]

# Data wrangling

In [7]:
# Indexing each pc task in each job
df.at[:,'indexed_pc'] = df['pc_text'].map(lambda pc: [(x[0],str(y)) for x,y in np.ndenumerate(pc)] if pc is not None else None)

# Processing

In [8]:
# Creating batches
def batch_skill (skill_list, batch_size = 20):
    l = len(skill_list)
    for idx in range (0,l,batch_size):
        yield skill_list[idx:min(idx+batch_size,l)]

In [9]:
# Setting up Groq
api_cycle = itertools.cycle(api_keys)
timeout_secs = 6*1000
client = genai.Client(api_key=next(api_cycle), http_options= HttpOptions(timeout=timeout_secs))

In [10]:
# Function to change client if rate limit is hit
def change_client ():
    global client
    client = genai.Client(api_key = next(api_cycle), http_options= HttpOptions(timeout=timeout_secs))
    print("Changed to another API key")

In [ ]:
# Create unprocessed rows.txt if not there
file_name = 'gemma_unprocessed_rows.txt'
if not Path(file_name).exists(): 
    with open (file_name, 'w') as f:
        print ("Empty txt file for storing index of unprocessed rows created.")
        pass
else:
    print(f"File {file_name} already exists")

# Create processed rows.txt if not there
file_name = 'gemma_processed_rows.txt'
if not Path(file_name).exists(): 
    with open (file_name, 'w') as f:
        print ("Empty txt file for storing index of processed rows created.")
        pass
else:
    print(f"File {file_name} already exists")

File gemini_unprocessed_rows.txt already exists
File gemini_processed_rows.txt already exists


In [12]:
# Initialising stop
stop = False

In [13]:
# Processing a single job
def clean_single_job (row_index: int, batch_size = 20,table = df, max_attempts = 5, model_name = 'gemma-3-12b-it'):
    
    # Stop flag 
    global stop

    # Defining metadata
    job_name = table.loc[row_index, 'Title']
    job_description = table.loc[row_index, 'Description']
    sector_name = table.loc[row_index, 'Sector Name']
    skill_list = table.loc[row_index, 'indexed_pc']

    # Initialising job level clean skills list
    clean_skills_job = []
    
    # Creating batches
    skill_batches = [x for x in batch_skill(skill_list = skill_list, batch_size=batch_size)]

    # Processing batches
    for i,batch in enumerate(skill_batches):
        
        print(f"processing batch {i+1} of {len(skill_batches)}")
        # Defining system prompt
        system_prompt = """
                            You are a JSON generator. Respond ONLY with valid JSON. 
                            - Output ONLY the raw JSON with no markdown formatting, no code blocks, no ```json markers, and no additional text.
                            - Do NOT provide any explanation or extra text.

                            For each item, return [unique_id, flag, original_description, corrected_description]:
                            - If skill description is complete: [unique_id, 0, "original_description", null]
                            - If skill description is incomplete: [unique_id, 1, "original_description", "corrected description"]

                            Rules:
                            - Input will be a list of tuples: (unique_id, skill description).
                            - Use null (not None) for empty values.
                            - Use unique_id as provided.
                            - Use double quotes for all strings.
                            - No explanations, no extra text.
                            - Length of output must equal length of input list.
                            - Ensure valid JSON syntax.

                            Output format:
                            {
                            "results": [
                                [unique_id, 0, "some description", null],
                                [unique_id, 1, "incomplete desc", "fixed version"]
                                        ]
                            }

                            Incorrect format (do not do this):
                            ```json
                            {
                            "results": [
                                [unique_id, 0, "some description", null],
                                [unique_id, 1, "incomplete desc", "fixed version"]
                                        ]
                            }```
                            """
        # Defining user prompt
        user_prompt = f"""
                Skills to analyze: {batch}
                Job details: 
                    job name: {job_name}
                    job description: {job_description} 
                    sector name: {sector_name}
                Return corrected versions of the skill descriptions in the instructed format.   
                """
        
        # Don't process the batch if daily api limit is hit, i.e., stop = True
        if stop:
            break

        # Make LLM call
        current_attempt = 1
        while current_attempt <=max_attempts:
            
            # Initialising batch list
            clean_skills_batch = None

            # Chat completion
            try:
                chat_completion = client.models.generate_content(
                                    contents= system_prompt + user_prompt,
                                    model = model_name,
                                )
                
                if chat_completion.text:
                            # Remove markdown elements put by gemini SDK
                            results = re.sub(r'^```json\s*|\s*```$', '', chat_completion.text, flags=re.MULTILINE)
                            results = json.loads(results)

            # APIError
            except google.genai.errors.APIError as e:
                if e.code == 503:
                    print(f"Api error 503 occurred. Trying after 20 seconds. Error: {e}")
                    current_attempt +=1
                    time.sleep(20)
                    print(f"Attempt number: {current_attempt} for batch {i+1} of job {row_index}")
                    current_attempt+=1
                    continue
                elif e.code == 429:
                    print(f"API error 429 occured. Daily limit exhausted. Changing API key. Error: {e}")
                    current_attempt +=1
                    time.sleep(2)
                    print(f"Attempt number: {current_attempt} for batch {i+1} of job {row_index}")
                    change_client()
                    continue

            
            # All other exceptions
            except Exception as e:
                print(f"Unexpected exception occurred: {e}")
                current_attempt +=1
                print(f"Attempt number: {current_attempt} for batch {i+1} of job {row_index}")
                time.sleep(2)
                continue

            if results:
                # If LLM returns a dictionary
                if isinstance(results, dict) and "results" in results:
                    if (len(results["results"]) == len(batch)):                         # Checks if returned size is same as batch size
                        if all(isinstance(x,list) for x in results["results"]):          # Checks if a list is returned
                            if all(x is not None for x in results["results"]):          # Checks if No skill is returned as None
                                if all(len(x) == 4 for x in results["results"]):        # Checks if each array is of size 4
                                    clean_skills_batch = results["results"]
                                    clean_skills_job.extend(clean_skills_batch)
                                    break
                            else:
                                print(f"None value returned for some skill in batch {i+1} fpr job {row_index}. Trying again")
                                current_attempt+=1
                        else:
                            print(f"Some skill not returned as a valid list in batch {i+1} for job {row_index}. Trying again")
                            current_attempt+=1
                    else:
                        print(f"Skills length {len(results['results'])} not equal to {len(batch)}, trying again.")
                        current_attempt+=1
                
                # If llm returns a list of list
                elif isinstance(results, list):
                    if len(results) == len(batch):                                  # Checks if returned size is same as batch size 
                        if all(isinstance(x,list) for x in results):                # Checks if a list is returned
                            if all(x is not None for x in results):                 # Checks if No skill is returned as None
                                if all(len(x) == 4 for x in results):               # Checks if each array is of size 4
                                    clean_skills_batch = results
                                    clean_skills_job.extend(clean_skills_batch)
                                    break
                            else:
                                print(f"None value returned for some skill in batch {i+1} fpr job {row_index}. Trying again")
                                current_attempt+=1
                        else:
                            print(f"Some skill not returned as a valid list in batch {i+1} for job {row_index}. Trying again")
                            current_attempt+=1
                    else:
                        print(f"Skills length {len(results)} not equal to {len(batch)}, trying again.")
                        current_attempt+=1

                
        if clean_skills_batch is None:
            print(f"Processing for batch {i+1} in job {row_index} failed. Not processing this job")
            return None
        
    # Final return
    if clean_skills_job:
        return clean_skills_job
    else:
        print(f"Processing for job with index {row_index} failed.")
        return None

In [67]:
# Processing data

## Initialising empty list of unprocessed rows
unprocessed_rows = []
processed_rows = []


## Rows to process
last_processed_row = 1927 # Put -1 when starting fresh
start_row = last_processed_row + 1
outer_batch_size = 200
batch_df = df.loc[start_row:start_row + (outer_batch_size-1)].copy()
batch_df['processed_pc'] = None

# Processing for each row in batch df

stop = False
for index, row in batch_df.iterrows():
    
    # Initializing stop flag for API Rate Limit
    print("*"*20,f"Processing row {index}","*"*20)
    
    # Processing rows with pc text
    if row['indexed_pc']:
        clean_pc = clean_single_job(index)
        if clean_pc:
            batch_df.at[index, 'processed_pc'] = clean_pc.to_list if isinstance(clean_pc,np.ndarray) else clean_pc
            processed_rows.append(index)
        else:
            if stop == False:
                batch_df.at[index, 'processed_pc'] = None
                unprocessed_rows.append(index)
            elif stop == True:
                print(f"Api limit over for the day, could not process row: {index}", "*"*20)
                break                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       
    
    # Processing rows without pc text
    else:
        batch_df.loc[index, 'processed_pc'] = None
        processed_rows.append(index)

print(f"Rows processed with index: {processed_rows}")

******************** Processing row 1928 ********************
processing batch 1 of 9
processing batch 2 of 9
processing batch 3 of 9
processing batch 4 of 9
processing batch 5 of 9
Unexpected exception occurred: The read operation timed out
Attempt number: 2 for batch 5 of job 1928
processing batch 6 of 9
Unexpected exception occurred: The read operation timed out
Attempt number: 2 for batch 6 of job 1928
Unexpected exception occurred: The read operation timed out
Attempt number: 3 for batch 6 of job 1928
Unexpected exception occurred: The read operation timed out
Attempt number: 4 for batch 6 of job 1928
Unexpected exception occurred: The read operation timed out
Attempt number: 5 for batch 6 of job 1928
Unexpected exception occurred: The read operation timed out
Attempt number: 6 for batch 6 of job 1928
Processing for batch 6 in job 1928 failed. Not processing this job
******************** Processing row 1929 ********************
processing batch 1 of 5
processing batch 2 of 5
proce

In [ ]:
# Saving batch
def save_batch (batch_id, table,folder = "llm_batches"):    
    
    # Make directory if it doesnt exist
    os.makedirs(folder, exist_ok=True)

    # Saving data frame
    file_name = f"{folder}/batch_{batch_id}.parquet"
    if not Path(file_name).exists():                                                                    
        for col in ['pc_text', 'indexed_pc', 'processed_pc']:
            table = table.copy()
            table [col] = table [col].apply(lambda x: json.dumps(x.tolist() if isinstance(x, np.ndarray) else x) 
                                                if isinstance(x, (list, np.ndarray)) else x)
        table.to_parquet(file_name, index=True)
    else:
        print("Chosen batch id already exists")

In [68]:
len(unprocessed_rows)

17

In [69]:
# Storing batch file
save_batch(21, batch_df)

In [ ]:
# Adding indexes of unprocessed rows
with open("gemma_unprocessed_rows.txt", "r") as f:
    existing_rows = set(int(line.strip()) for line in f)


with open("gemma_unprocessed_rows.txt", "a") as f:
    for x in unprocessed_rows:
        if x not in existing_rows:
            f.write(f"{x}\n")

In [ ]:
# Adding indexes of processed rows
with open("gemma_processed_rows.txt", "r") as f:
    existing_rows = set(int(line.strip()) for line in f)


with open("gemma_processed_rows.txt", "a") as f:
    for x in processed_rows:
        if x not in existing_rows:
            f.write(f"{x}\n")